# Partie 3 : évaluer les approches sur un vrai corpus français

**Objectif** : mesurer, sur des milliers d'avis réels et déjà annotés, ce que les parties 1 et 2 n'ont pu que montrer sur quelques exemples.

**Corpus** : critiques de films **Allociné** (avis en français, étiquetés positif ou négatif), en accès libre sur Hugging Face.

**Les approches comparées**, sur exactement les mêmes avis :

| # | Approche | D'où elle vient |
|---|---|---|
| 1 | VADER + TextBlob | Partie 1 (outils conçus pour l'anglais) |
| 2 | Lexique français pondéré | Partie 2 (`EtiqueteurRegles`) |
| 3 | TF-IDF + Naive Bayes | Partie 2, entraîné cette fois sur un vrai volume de données |
| 4 | TF-IDF + régression logistique | Modèle supervisé classique |
| 5 | BERT multilingue pré-entraîné | Modèle de langage, utilisé **sans** réentraînement sur Allociné |

**Mesures** : taux de bonnes réponses (accuracy), F1-score, et **couverture** (part des avis pour lesquels la méthode ose trancher).

**Pour l'exécuter** : Google Colab, puis *Exécution → Modifier le type d'exécution → GPU* (facultatif, mais le modèle 5 sera plus rapide). Exécuter les cellules dans l'ordre.

In [ ]:
# Installation des bibliothèques (Google Colab)
!pip install -q datasets transformers vaderSentiment textblob scikit-learn

In [ ]:
import os, time, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

# Paramètres de l'expérience
SEED = 42          # pour des résultats reproductibles
N_TRAIN = 20000    # avis d'entraînement (méthodes 3 et 4)
N_TEST = 2000      # avis de test, communs à toutes les méthodes
random.seed(SEED); np.random.seed(SEED)
os.makedirs("images", exist_ok=True)
LABELS = {0: "négatif", 1: "positif"}
print("Configuration prête.")

## 1. Chargement du corpus Allociné

In [ ]:
from datasets import load_dataset

def charger_allocine():
    """Charge le corpus Allociné ; essaie les deux noms sous lesquels il est publié."""
    for nom in ["tblard/allocine", "allocine"]:
        try:
            ds = load_dataset(nom)
            print(f"Corpus chargé : {nom}")
            return ds
        except Exception as e:
            print(f"Échec avec '{nom}' : {e}")
    raise RuntimeError("Impossible de charger le corpus Allociné.")

ds = charger_allocine()
train = ds["train"].shuffle(seed=SEED).select(range(N_TRAIN)).to_pandas()
test = ds["test"].shuffle(seed=SEED).select(range(N_TEST)).to_pandas()

print(f"Entraînement : {len(train)} avis | Test : {len(test)} avis")
print("Répartition des classes (test) :", test["label"].map(LABELS).value_counts().to_dict())
test.head(3)

## 2. Fonction d'évaluation commune

Certaines méthodes peuvent **s'abstenir** (score nul, avis classé « neutre »). Le corpus ne contient que des avis positifs ou négatifs : une abstention est donc comptée comme une erreur, et on mesure à part la **couverture**.

In [ ]:
RESULTATS = []
PREDICTIONS = {}

def evaluer(nom, y_true, y_pred, duree):
    """y_pred : 0, 1 ou None (abstention)."""
    y_true = np.asarray(y_true)
    pred = np.array([-1 if p is None else p for p in y_pred])
    couverture = float(np.mean(pred != -1))
    res = {
        "méthode": nom,
        "accuracy": accuracy_score(y_true, pred),
        "f1_macro": f1_score(y_true, pred, labels=[0, 1], average="macro", zero_division=0),
        "couverture": couverture,
        "accuracy_quand_elle_tranche": accuracy_score(y_true[pred != -1], pred[pred != -1]) if couverture > 0 else float("nan"),
        "durée (s)": round(duree, 1),
    }
    RESULTATS.append(res)
    PREDICTIONS[nom] = pred
    print(f"{nom:<38} accuracy={res['accuracy']:.3f}  F1={res['f1_macro']:.3f}  couverture={couverture:.1%}")
    return res

## 3. Méthode 1 : VADER + TextBlob (partie 1)

In [ ]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from textblob import TextBlob

vader = SentimentIntensityAnalyzer()

def predire_vader_textblob(texte):
    # Même calcul que DocumentSentimentAnalyzer (partie 1) : 0,6 x VADER + 0,4 x TextBlob
    score = 0.6 * vader.polarity_scores(texte)["compound"] + 0.4 * TextBlob(texte).sentiment.polarity
    if score >= 0.1:
        return 1
    if score <= -0.1:
        return 0
    return None  # « Neutre » dans la partie 1 : abstention

t0 = time.time()
pred = [predire_vader_textblob(t) for t in test["review"]]
evaluer("1. VADER + TextBlob (partie 1)", test["label"], pred, time.time() - t0)

## 4. Méthode 2 : lexique français pondéré (partie 2)

In [ ]:
class EtiqueteurRegles:
    """Repris tel quel de la partie 2."""
    def __init__(self):
        self.mots_positifs = {'bon': 1, 'délicieux': 2, 'impeccable': 2, 'excellent': 3,
                              'génial': 2, 'super': 2, 'parfait': 3,
                              'fantastique': 3, 'succulent': 2, 'aimable': 1}
        self.mots_negatifs = {'horrible': 3, 'cher': 1, 'désagréable': 2,
                              'interminable': 2, 'catastrophique': 3,
                              'mauvais': 2, 'nul': 2, 'lent': 1, 'arnaqué': 3, 'décevant': 2}

    def score(self, texte):
        t = texte.lower()
        return (sum(p for m, p in self.mots_positifs.items() if m in t)
                - sum(p for m, p in self.mots_negatifs.items() if m in t))

regles = EtiqueteurRegles()

def predire_lexique(texte):
    s = regles.score(texte)
    if s > 2:
        return 1
    if s <= -2:
        return 0
    return None  # « neutre » : abstention

t0 = time.time()
pred = [predire_lexique(t) for t in test["review"]]
evaluer("2. Lexique français (partie 2)", test["label"], pred, time.time() - t0)

## 5. Méthodes 3 et 4 : modèles supervisés TF-IDF

Dans la partie 2, le Naive Bayes était entraîné sur 6 phrases. Ici, il apprend sur **20 000 avis réels**.

In [ ]:
vectoriseur = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=100_000, sublinear_tf=True)
X_train = vectoriseur.fit_transform(train["review"])
X_test = vectoriseur.transform(test["review"])

for nom, modele in [("3. TF-IDF + Naive Bayes", MultinomialNB()),
                    ("4. TF-IDF + régression logistique", LogisticRegression(max_iter=1000, C=4))]:
    t0 = time.time()
    modele.fit(X_train, train["label"])
    pred = modele.predict(X_test)
    evaluer(nom, test["label"], list(pred), time.time() - t0)

# Les mots qui pèsent le plus dans la régression logistique : on regarde ce que le modèle a appris
mots = np.array(vectoriseur.get_feature_names_out())
coefs = modele.coef_[0]
print("\nMots les plus positifs :", ", ".join(mots[np.argsort(coefs)[-15:]][::-1]))
print("Mots les plus négatifs :", ", ".join(mots[np.argsort(coefs)[:15]]))
lr = modele  # conservé pour l'analyse des erreurs

## 6. Méthode 5 : modèle de langage pré-entraîné

Modèle BERT multilingue entraîné sur des avis produits notés de 1 à 5 étoiles (`nlptown/bert-base-multilingual-uncased-sentiment`). Il n'a **jamais vu Allociné** : on teste sa capacité à généraliser à un nouveau type d'avis.

Règle de décision : positif si la probabilité des notes 4-5 dépasse celle des notes 1-2.

In [ ]:
import torch
from transformers import pipeline

clf = pipeline("text-classification",
               model="nlptown/bert-base-multilingual-uncased-sentiment",
               top_k=None, truncation=True, max_length=512,
               device=0 if torch.cuda.is_available() else -1)
print("GPU utilisé" if torch.cuda.is_available() else "CPU utilisé (quelques minutes)")

def decision_etoiles(scores):
    p = {d["label"]: d["score"] for d in scores}
    positif = p.get("4 stars", 0) + p.get("5 stars", 0)
    negatif = p.get("1 star", 0) + p.get("2 stars", 0)
    return 1 if positif > negatif else 0

t0 = time.time()
sorties = clf(list(test["review"]), batch_size=32)
pred = [decision_etoiles(s) for s in sorties]
evaluer("5. BERT multilingue pré-entraîné", test["label"], pred, time.time() - t0)

## 7. Comparaison des méthodes

In [ ]:
df_res = pd.DataFrame(RESULTATS).sort_values("accuracy", ascending=False).reset_index(drop=True)
df_res.to_csv("resultats_comparaison.csv", index=False)
display(df_res.style.format({"accuracy": "{:.1%}", "f1_macro": "{:.3f}",
                             "couverture": "{:.1%}", "accuracy_quand_elle_tranche": "{:.1%}"}))

fig, ax = plt.subplots(figsize=(11, 5))
x = np.arange(len(df_res))
ax.bar(x - 0.2, df_res["accuracy"] * 100, 0.4, label="Accuracy", color="#1F3A5F")
ax.bar(x + 0.2, df_res["couverture"] * 100, 0.4, label="Couverture", color="#C98A12")
for i, v in enumerate(df_res["accuracy"] * 100):
    ax.text(i - 0.2, v + 1, f"{v:.0f} %", ha="center", fontweight="bold")
ax.axhline(50, ls="--", color="grey", lw=1)
ax.text(len(df_res) - 0.5, 51.5, "hasard (50 %)", ha="right", color="grey")
ax.set_xticks(x)
ax.set_xticklabels([n.split(". ", 1)[1] for n in df_res["méthode"]], rotation=15, ha="right")
ax.set_ylim(0, 105)
ax.set_ylabel("%")
ax.set_title(f"Analyse de sentiments sur {N_TEST} critiques Allociné", fontweight="bold")
ax.legend()
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig("images/comparaison_methodes.png", dpi=130)
plt.show()

In [ ]:
fig, axes = plt.subplots(1, len(PREDICTIONS), figsize=(4 * len(PREDICTIONS), 3.6))
for ax, (nom, pred) in zip(axes, PREDICTIONS.items()):
    cm = confusion_matrix(test["label"], pred, labels=[0, 1, -1])[:2]  # lignes : vrai négatif / positif
    sns.heatmap(cm, annot=True, fmt="d", cbar=False, cmap="Blues", ax=ax,
                xticklabels=["négatif", "positif", "abstention"], yticklabels=["négatif", "positif"])
    ax.set_title(nom.split(". ", 1)[1], fontsize=10)
    ax.set_xlabel("prédit")
    ax.set_ylabel("réel")
plt.tight_layout()
plt.savefig("images/matrices_confusion.png", dpi=130)
plt.show()

## 8. Analyse des erreurs du meilleur modèle supervisé

In [ ]:
erreurs = test[PREDICTIONS["4. TF-IDF + régression logistique"] != test["label"]].copy()
erreurs["réel"] = erreurs["label"].map(LABELS)
print(f"{len(erreurs)} erreurs sur {len(test)} avis ({len(erreurs) / len(test):.1%})\n")
for _, r in erreurs.sample(min(8, len(erreurs)), random_state=SEED).iterrows():
    print(f"[réel : {r['réel']}] {r['review'][:220]}...\n")

## 9. Retour aux cas difficiles des parties 1 et 2

Les avis de la partie 1 (e-commerce) et les exemples d'ironie de la partie 2 (restaurant) : comment chaque méthode s'en sort-elle ?

In [ ]:
cas = pd.DataFrame([
    ("Le téléphone est excellent, très rapide et l'autonomie dure toute la journée. Cependant, la caméra est décevante et le prix trop élevé à mon goût. Service client impeccable!", "mitigé"),
    ("Livraison rapide mais produit arrivé endommagé. Je suis très frustré et déçu par cette expérience.", "négatif"),
    ("Rien à redire! Produit parfait, emballage soigné, livraison express. Je recommande vivement!", "positif"),
    ("La qualité sonore est exceptionnelle mais l'application mobile bug constamment. Support technique lent à répondre.", "mitigé"),
    ("Bof... produit correct sans plus. Le rapport qualité-prix est moyen. Pas sûr de recommander.", "négatif"),
    ("Le 'service impeccable' était en fait une attente de 45 minutes.", "négatif (ironie)"),
    ("Le gâteau au chocolat était à tomber par terre, littéralement !", "positif (expression)"),
    ("Super, j'ai attendu 2h pour être servi !", "négatif (ironie)"),
    ("Je ne reviendrai JAMAIS ici, sauf si c'est gratuit.", "négatif"),
    ("Ce n'est pas mauvais du tout, je recommande.", "positif (négation)"),
], columns=["avis", "attendu"])

fmt = lambda p: "abstention" if p is None else LABELS[p]
cas["VADER+TextBlob"] = [fmt(predire_vader_textblob(t)) for t in cas["avis"]]
cas["Lexique FR"] = [fmt(predire_lexique(t)) for t in cas["avis"]]
cas["TF-IDF + rég. log."] = [LABELS[p] for p in lr.predict(vectoriseur.transform(cas["avis"]))]
cas["BERT multilingue"] = [LABELS[decision_etoiles(s)] for s in clf(list(cas["avis"]))]
pd.set_option("display.max_colwidth", 70)
cas

## 10. Conclusion

Résultats obtenus sur 2 000 critiques Allociné (exécution sur GPU, Google Colab) :

| Méthode | Accuracy | F1 | Couverture | Précision quand elle tranche |
|---|---|---|---|---|
| TF-IDF + régression logistique | **93,1 %** | 0,931 | 100 % | 93,1 % |
| TF-IDF + Naive Bayes | 90,8 % | 0,908 | 100 % | 90,8 % |
| BERT multilingue pré-entraîné | 90,2 % | 0,902 | 100 % | 90,2 % |
| VADER + TextBlob (partie 1) | 39,6 % | 0,477 | 65 % | 60,8 % |
| Lexique français (partie 2) | 21,1 % | 0,338 | 24 % | 86,6 % |

**1. Les outils conçus pour l'anglais ne conviennent pas au français.** Même quand VADER et TextBlob tranchent, ils n'ont raison que dans 60,8 % des cas, à peine mieux que le hasard. Le diagnostic de la partie 1 est confirmé.

**2. Le lexique français est précis mais trop étroit.** Il a raison dans 86,6 % des cas quand il tranche, mais ne reconnaît que 24 % des avis : 20 mots ne suffisent pas à couvrir la diversité du vocabulaire. C'est le compromis entre précision et couverture.

**3. Le volume de données change tout.** Le Naive Bayes de la partie 2, entraîné sur 20 000 avis au lieu de 6 phrases, atteint 90,8 %.

**4. Le meilleur modèle est aussi le plus simple** : TF-IDF + régression logistique, 93,1 %, en quelques secondes d'entraînement.

**5. Le modèle pré-entraîné généralise très bien** : 90,2 % sans avoir jamais vu Allociné, mais il est nettement plus lent (53 s sur GPU pour 2 000 avis).

**6. Cas difficiles** : sur nos deux exemples d'ironie, la régression logistique répond juste et BERT se trompe ; aucun modèle ne comprend l'expression « à tomber par terre ». Avec si peu d'exemples, c'est une observation, pas une conclusion générale : l'ironie et les expressions idiomatiques restent les cas les plus difficiles.

**Pour aller plus loin** : affiner un modèle français (CamemBERT) sur Allociné, et constituer un jeu d'avis ironiques annotés pour évaluer spécifiquement ces cas.

In [ ]:
# Résumé chiffré, à copier dans le README
for _, r in df_res.iterrows():
    print(f"- {r['méthode']} : accuracy {r['accuracy']:.1%}, F1 {r['f1_macro']:.3f}, couverture {r['couverture']:.0%}")